In [1]:
import logging
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass

import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder


@dataclass
class _SimilarityConfig:
    model_name: str = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
    cross_encoder_name: str = 'cross-encoder/stsb-distilroberta-base'
    collection_name: str = 'article_similarity'
    chroma_db_path: str = './chroma_db'
    batch_size: int = 32
    default_limit: int = 10
    default_similarity_threshold: float = 0.7
    rerank_top_k: int = 100
    device: Optional[str] = None

    def __post_init__(self):
        if self.device is None:
            import torch
            if torch.cuda.is_available():
                self.device = 'cuda'
            elif torch.backends.mps.is_available():
                self.device = 'mps'
            else:
                self.device = 'cpu'


class _ArticleSimilarity:
    """Internal engine – not exposed to backend."""
    
    def __init__(self, config: _SimilarityConfig):
        self.config = config
        self.client = None
        self.collection = None
        self.bi_encoder = None
        self.cross_encoder = None
        self._init_client()
        self._init_bi_encoder()

    def _init_client(self):
        self.client = chromadb.PersistentClient(path=self.config.chroma_db_path)
        self.collection = self.client.get_or_create_collection(
            name=self.config.collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        logging.info(f"ChromaDB collection '{self.config.collection_name}' ready. "
                     f"Current count: {self.collection.count()}")

    def _init_bi_encoder(self):
        logging.info(f"Loading bi‑encoder model {self.config.model_name} on {self.config.device}...")
        self.bi_encoder = SentenceTransformer(self.config.model_name, device=self.config.device)

    def _init_cross_encoder(self):
        if not self.config.cross_encoder_name:
            raise RuntimeError("Cross‑encoder name not provided in configuration.")
        if self.cross_encoder is None:
            logging.info(f"Loading cross‑encoder model {self.config.cross_encoder_name}...")
            self.cross_encoder = CrossEncoder(self.config.cross_encoder_name, device=self.config.device)

    def _embed(self, texts: List[str]) -> List[List[float]]:
        return self.bi_encoder.encode(
            texts,
            batch_size=self.config.batch_size,
            convert_to_tensor=False,
            show_progress_bar=False
        ).tolist()

    def _combine_text(self, abstract: str, keywords: str) -> str:
        combined = f"{abstract} [KEYWORDS] {keywords}" if keywords else abstract
        return combined.strip()

    def add_article(self, article_id: str, abstract: str, keywords: str = "", 
                    category: str = "", metadata: Optional[Dict] = None) -> None:
        self.add_articles([(article_id, abstract, keywords, category, metadata)])

    def add_articles(self, articles: List[Tuple[str, str, str, str, Optional[Dict]]]) -> None:
        if not articles:
            return

        ids, combined_texts, metadatas = [], [], []
        for art in articles:
            a_id, abstract, keywords, category, extra_meta = art
            ids.append(a_id)
            combined = self._combine_text(abstract, keywords)
            combined_texts.append(combined)
            meta = (extra_meta or {}).copy()
            meta.update({'category': category, 'abstract': abstract, 'keywords': keywords})
            metadatas.append(meta)

        logging.info(f"Encoding {len(articles)} articles...")
        embeddings = self._embed(combined_texts)

        self.collection.add(
            ids=ids,
            embeddings=embeddings,
            metadatas=metadatas,
            documents=combined_texts
        )
        logging.info(f"Added/updated {len(articles)} articles.")

    def delete_article(self, article_id: str) -> None:
        self.collection.delete(ids=[article_id])
        logging.info(f"Deleted article {article_id}.")

    def get_article(self, article_id: str) -> Optional[Dict]:
        result = self.collection.get(ids=[article_id], include=['metadatas', 'documents'])
        if not result['ids']:
            return None
        meta = result['metadatas'][0]
        return {
            'id': result['ids'][0],
            'abstract': meta.get('abstract', ''),
            'keywords': meta.get('keywords', ''),
            'category': meta.get('category', ''),
            'metadata': meta,
            'combined_text': result['documents'][0]
        }

    def update_article(self, article_id: str, abstract: Optional[str] = None, 
                       keywords: Optional[str] = None, category: Optional[str] = None, 
                       metadata: Optional[Dict] = None) -> None:
        existing = self.get_article(article_id)
        if not existing:
            raise ValueError(f"Article {article_id} not found.")

        text_changed = (abstract is not None and abstract != existing['abstract']) or \
                       (keywords is not None and keywords != existing['keywords'])

        if text_changed:
            new_abstract = abstract if abstract is not None else existing['abstract']
            new_keywords = keywords if keywords is not None else existing['keywords']
            new_cat = category if category is not None else existing['category']
            new_meta = (metadata or existing['metadata']).copy()
            self.add_article(article_id, new_abstract, new_keywords, new_cat, new_meta)
        else:
            new_meta = existing['metadata'].copy()
            if category is not None: new_meta['category'] = category
            if metadata: new_meta.update(metadata)
            self.collection.update(ids=[article_id], metadatas=[new_meta])

    def find_similar_by_id_bi(self, article_id: str, limit: Optional[int] = None, 
                              similarity_threshold: Optional[float] = None, 
                              exclude_self: bool = True, same_category_only: bool = True) -> List[Dict]:
        limit = limit or self.config.default_limit
        threshold = similarity_threshold or self.config.default_similarity_threshold

        result = self.collection.get(ids=[article_id], include=['embeddings', 'metadatas'])
        if not result['ids']:
            raise ValueError(f"Article {article_id} not found.")

        query_emb, category = result['embeddings'][0], result['metadatas'][0].get('category', '')
        where = {"category": {"$eq": category}} if same_category_only and category else None

        query_res = self.collection.query(
            query_embeddings=[query_emb],
            n_results=limit + (1 if exclude_self else 0),
            where=where,
            include=['metadatas', 'distances']
        )

        output = []
        for i in range(len(query_res['ids'][0])):
            cid = query_res['ids'][0][i]
            if exclude_self and cid == article_id: continue
            if len(output) >= limit: break
            sim = max(0.0, min(1.0, 1.0 - query_res['distances'][0][i]))
            if sim < threshold: continue
            meta = query_res['metadatas'][0][i]
            output.append({
                'id': cid, 'abstract': meta.get('abstract', ''), 
                'keywords': meta.get('keywords', ''), 'category': meta.get('category', ''), 
                'similarity': sim
            })
        return output

    def _rerank_candidates(self, query_article_id: str, candidate_ids: List[str]) -> List[Tuple[str, float]]:
        if not self.cross_encoder:
            self._init_cross_encoder()

        query_article = self.get_article(query_article_id)
        if not query_article or not candidate_ids:
            return []

        res = self.collection.get(ids=candidate_ids, include=['documents'])
        doc_map = dict(zip(res['ids'], res['documents']))
        
        valid_pairs, valid_ids = [], []
        for cid in candidate_ids:
            if cid in doc_map:
                valid_pairs.append((query_article['combined_text'], doc_map[cid]))
                valid_ids.append(cid)

        if not valid_pairs:
            return []
        scores = self.cross_encoder.predict(valid_pairs, batch_size=self.config.batch_size)
        
        scored = [(valid_ids[i], float(scores[i])) for i in range(len(valid_ids))]
        return sorted(scored, key=lambda x: x[1], reverse=True)

    def rerank_similar(self, article_id: str, top_k: Optional[int] = None, 
                       same_category_only: bool = True) -> List[Tuple[str, float]]:
        top_k = top_k or self.config.rerank_top_k
        candidates_bi = self.find_similar_by_id_bi(
            article_id, limit=top_k, similarity_threshold=0.0, 
            exclude_self=True, same_category_only=same_category_only
        )
        candidate_ids = [c['id'] for c in candidates_bi]
        return self._rerank_candidates(article_id, candidate_ids)

    def find_similar_by_id(self, article_id: str, limit: Optional[int] = None, 
                           similarity_threshold: Optional[float] = None, 
                           exclude_self: bool = True, same_category_only: bool = True, 
                           use_cross_encoder: bool = False, force_rerank: bool = False, 
                           offset: int = 0) -> List[Dict]:
        limit = limit or self.config.default_limit
        threshold = similarity_threshold or self.config.default_similarity_threshold

        if not use_cross_encoder or not self.config.cross_encoder_name:
            return self.find_similar_by_id_bi(article_id, limit, threshold, exclude_self, same_category_only)

        reranked_list = self.rerank_similar(article_id, same_category_only=same_category_only)
        paginated = reranked_list[offset : offset + limit]

        output = []
        if paginated:
            ids_to_fetch = [item[0] for item in paginated]
            res = self.collection.get(ids=ids_to_fetch, include=['metadatas'])
            meta_map = dict(zip(res['ids'], res['metadatas']))

            for cid, score in paginated:
                if score < threshold:
                    continue
                meta = meta_map.get(cid, {})
                output.append({
                    'id': cid, 'abstract': meta.get('abstract', ''), 
                    'keywords': meta.get('keywords', ''), 'category': meta.get('category', ''), 
                    'similarity': score
                })
        return output


class Article:
    """Simple data class for article input."""
    def __init__(self, article_id: str, abstract: str, keywords: str = "", category: str = ""):
        self.id = article_id
        self.abstract = abstract
        self.keywords = keywords
        self.category = category

class SimilarityService:
    def __init__(self, db_path: str = './chroma_db', batch_size: int = 32, 
                 threshold: float = 0.7, rerank_top_k: int = 100):
        """
        Initialize the similarity service with optional parameters.
        If you need to change model names, modify the config inside this method.
        """
        # Internal config – modify defaults here if needed
        config = _SimilarityConfig(
            model_name='sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
            cross_encoder_name='cross-encoder/stsb-distilroberta-base',
            collection_name='article_similarity',
            chroma_db_path=db_path,
            batch_size=batch_size,
            default_similarity_threshold=threshold,
            rerank_top_k=rerank_top_k,
            device="cpu"
        )
        self._engine = _ArticleSimilarity(config)

    def add(self, article_id: str, abstract: str, keywords: str = "", category: str = "") -> None:
        """Add a single article."""
        self._engine.add_article(article_id, abstract, keywords, category)

    def add_batch(self, articles: List[Article]) -> None:
        """
        Add multiple articles.
        Each article is an Article object with fields: id, abstract, keywords, category.
        """
        converted = [(a.id, a.abstract, a.keywords, a.category, None) for a in articles]
        self._engine.add_articles(converted)

    def similar(self, article_id: str, limit: int = 10,
                same_category: bool = True, use_cross_encoder: bool = False) -> List[Dict]:
        """
        Get similar articles.
        Returns list of dicts with keys: id, abstract, keywords, category, similarity.
        """
        return self._engine.find_similar_by_id(
            article_id, limit=limit, same_category_only=same_category,
            use_cross_encoder=use_cross_encoder
        )

    def get(self, article_id: str) -> Optional[Dict]:
        """
        Retrieve an article by ID.
        Returns dict with keys: id, abstract, keywords, category.
        """
        article = self._engine.get_article(article_id)
        if article is None:
            return None
        return {
            'id': article['id'],
            'abstract': article['abstract'],
            'keywords': article['keywords'],
            'category': article['category']
        }

    def update(self, article_id: str, abstract: Optional[str] = None,
               keywords: Optional[str] = None, category: Optional[str] = None) -> None:
        """Update article fields."""
        self._engine.update_article(article_id, abstract=abstract, keywords=keywords, category=category)

    def delete(self, article_id: str) -> None:
        """Delete an article."""
        self._engine.delete_article(article_id)


# ----------------------------------------------------------------------
# Test script (optional)
# ----------------------------------------------------------------------
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)

    # Create service with custom db path and threshold
    service = SimilarityService(db_path='./arabic_test_db', threshold=0.3, batch_size=5)

    # Sample Arabic articles using the Article class
    articles = [
        Article("art_1", "الذكاء الاصطناعي يغير مستقبل البرمجة بشكل جذري.", "تقنية, ذكاء اصطناعي", "technology"),
        Article("art_2", "تعلم لغة بايثون يعتبر خطوة أساسية للمبتدئين في البرمجة.", "برمجة, بايثون", "technology"),
        Article("art_3", "أهمية الحوسبة السحابية في تطوير التطبيقات الحديثة.", "تقنية, سحاب", "technology"),
        Article("art_4", "كيفية حماية البيانات الشخصية على الإنترنت من الاختراق.", "أمن سيبراني, خصوصية", "technology"),
        Article("art_5", "مستقبل الهواتف الذكية وتقنيات الشاشات القابلة للطي.", "جوالات, تقنية", "technology"),
        
        Article("art_6", "فوائد ممارسة الرياضة الصباحية على الصحة النفسية.", "رياضة, صحة", "health"),
        Article("art_7", "أهمية شرب الماء بكميات كافية خلال فصل الصيف.", "صحة, تغذية", "health"),
        Article("art_8", "نظام غذائي متوازن يساعد في تقليل خطر الإصابة بالسكري.", "تغذية, سكري", "health"),
        Article("art_9", "تأثير النوم المتقطع على التركيز والنشاط اليومي.", "نوم, صحة", "health"),
        Article("art_10", "طرق طبيعية لتقوية جهاز المناعة في الشتاء.", "مناعة, صحة", "health"),

        Article("art_11", "نتائج مباريات الدوري الإسباني يوم أمس.", "كرة قدم, رياضة", "sports"),
        Article("art_12", "انتقالات اللاعبين في سوق الانتقالات الصيفية الحالية.", "كرة قدم, انتقالات", "sports"),
        Article("art_13", "تطور رياضة التنس وزيادة شعبيتها في العالم العربي.", "تنس, رياضة", "sports"),
        Article("art_14", "أفضل التمارين لتقوية عضلات القلب والتحمل.", "لياقة, رياضة", "sports"),
        Article("art_15", "استعدادات المنتخبات لبطولة كأس العالم القادمة.", "كرة قدم, مونديال", "sports"),

        Article("art_16", "تأثير التغير المناخي على المدن الساحلية.", "بيئة, مناخ", "environment"),
        Article("art_17", "الطاقة المتجددة هي الحل الأمثل لأزمة الكهرباء.", "طاقة, بيئة", "environment"),
        Article("art_18", "زراعة الأشجار في المدن تقلل من التلوث البصري والجوي.", "بيئة, زراعة", "environment"),
        Article("art_19", "الاعتماد على السيارات الكهربائية لتقليل الانبعاثات.", "سيارات, بيئة", "environment"),
        Article("art_20", "أهمية تدوير النفايات البلاستيكية لحماية المحيطات.", "تدوير, بيئة", "environment"),
    ]

    print("\n--- Adding Arabic Articles to Database ---")
    service.add_batch(articles)

    target_id = "art_1"
    print(f"\n--- Finding articles similar to: '{target_id}' ---")
    results = service.similar(target_id, limit=3, same_category=True, use_cross_encoder=True)

    if not results:
        print("No similar articles found.")
    else:
        for i, res in enumerate(results, 1):
            print(f"{i}. [Score: {res['similarity']:.4f}] ID: {res['id']}")
            print(f"   Abstract: {res['abstract']}")
            print(f"   Category: {res['category']}")
            print("-" * 30)

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:root:ChromaDB collection 'article_similarity' ready. Current count: 20
INFO:root:Loading bi‑encoder model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 on cpu...
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
INFO:root:Encoding 20 articles...



--- Adding Arabic Articles to Database ---


INFO:root:Added/updated 20 articles.
INFO:root:Loading cross‑encoder model cross-encoder/stsb-distilroberta-base...



--- Finding articles similar to: 'art_1' ---


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1. [Score: 0.7874] ID: art_5
   Abstract: مستقبل الهواتف الذكية وتقنيات الشاشات القابلة للطي.
   Category: technology
------------------------------
2. [Score: 0.7779] ID: art_2
   Abstract: تعلم لغة بايثون يعتبر خطوة أساسية للمبتدئين في البرمجة.
   Category: technology
------------------------------
3. [Score: 0.7061] ID: art_3
   Abstract: أهمية الحوسبة السحابية في تطوير التطبيقات الحديثة.
   Category: technology
------------------------------
